In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset
from torchvision import datasets, transforms
from torchvision.models.resnet import ResNet, BasicBlock
from torchvision.models import resnet18, ResNet18_Weights
from torch.utils.data import DataLoader, Subset
import torch.nn.functional as F
import numpy as np
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt
import numpy as np
import copy

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


# Group 1

In [2]:
cat_class = 3
dog_class = 5

transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.4914, 0.4822, 0.4465],
                         std=[0.247, 0.243, 0.261])
])

train_dataset_full = datasets.CIFAR10(root='./cifar-10-batches-py', train=True,
                                      download=True, transform=transform)
test_dataset_full = datasets.CIFAR10(root='./cifar-10-batches-py', train=False,
                                     download=True, transform=transform)

Files already downloaded and verified
Files already downloaded and verified


In [3]:
def filter_cat_vs_dog(dataset, cat_class=3, dog_class=5):
    indices = [i for i, label in enumerate(dataset.targets) if label in [cat_class, dog_class]]
    new_targets = [1 if dataset.targets[i] == cat_class else 0 for i in indices]
    subset = Subset(dataset, indices)
    subset.targets = new_targets
    return subset

In [4]:
class CatDogDataset(Dataset):
    def __init__(self, subset):
        self.subset = subset

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        img, _ = self.subset[idx]
        label = self.subset.targets[idx]
        return img, label

In [5]:
train_dataset = CatDogDataset(filter_cat_vs_dog(train_dataset_full))
test_dataset = CatDogDataset(filter_cat_vs_dog(test_dataset_full))

In [6]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

print("train - test dataset ready.")

train - test dataset ready.


# Group 2

In [7]:
# --- Part 2: Configure Model ---
# Task 2: Load pretrained ResNet18
print("\nLoading pretrained ResNet18 model...")
model = resnet18(weights=ResNet18_Weights.DEFAULT)

# Task 3: Modify the final fully connected layer
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 2)
# Task 4: Freeze all layers except layer3, layer4, and fc
# First, freeze all parameters
for param in model.parameters():
    param.requires_grad = False

# Unfreeze only the layers for fine-tuning
print("Unfreezing layers for fine-tuning: layer3, layer4, and fc.")
for param in model.layer3.parameters():
    param.requires_grad = True
for param in model.layer4.parameters():
    param.requires_grad = True
for param in model.fc.parameters():
    param.requires_grad = True

#Verification(rishab likes goth chicks)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("\nModel setup is complete.")
print(f"  Total parameters:     {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")
print("\nNotebook is now ready for Group 3 (Fine-Tuning).")


Loading pretrained ResNet18 model...
Unfreezing layers for fine-tuning: layer3, layer4, and fc.

Model setup is complete.
  Total parameters:     11,177,538
  Trainable parameters: 10,494,466

Notebook is now ready for Group 3 (Fine-Tuning).


In [64]:
# --- Group 3: Fine-Tuning, Evaluation, and Feature Extraction ---

import copy
from tqdm import tqdm
import torch

model = model.to(device)

# Define loss and optimizer BEFORE training
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)

# Check for existing saved model
if os.path.exists("model.pth"):
    print("Found existing 'model.pth' — skipping training and loading saved weights.")
    model.load_state_dict(torch.load("model.pth", map_location=device))
    model.eval()

else:
    print("No saved model found — starting training from scratch.")
    
    # --- Early Stopping Setup ---
    patience = 9
    best_val_acc = 0.0
    best_model_wts = copy.deepcopy(model.state_dict())
    counter = 0
    num_epochs = 15

    for epoch in range(num_epochs):
        print(f"\nEpoch [{epoch+1}/{num_epochs}]")
        print("-" * 30)
        
        # --- Training Phase ---
        model.train()
        running_loss, running_corrects, total = 0.0, 0, 0
        
        for inputs, labels in tqdm(train_loader, desc="Training", leave=False):
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            _, preds = torch.max(outputs, 1)
            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)
            total += labels.size(0)

        train_loss = running_loss / total
        train_acc = running_corrects.double() / total

        # --- Validation Phase ---
        model.eval()
        val_loss, val_corrects, val_total = 0.0, 0, 0
        
        with torch.no_grad():
            for inputs, labels in tqdm(test_loader, desc="Validating", leave=False):
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)

                _, preds = torch.max(outputs, 1)
                val_loss += loss.item() * inputs.size(0)
                val_corrects += torch.sum(preds == labels.data)
                val_total += labels.size(0)

        val_loss = val_loss / val_total
        val_acc = val_corrects.double() / val_total

        print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
        print(f"Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f}")

        # --- Early Stopping Check ---
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_wts = copy.deepcopy(model.state_dict())
            counter = 0
            print("Validation accuracy improved — saving best model.")
            torch.save(best_model_wts, "best_model.pth")
        else:
            counter += 1
            print(f"No improvement for {counter} epoch(s).")
            if counter >= patience:
                print("Early stopping triggered.")
                break

    # --- Load best weights after training ---
    model.load_state_dict(best_model_wts)
    print("\nTraining complete. Best Validation Accuracy: {:.4f}".format(best_val_acc))

    # --- Save the final best model ---
    torch.save(model.state_dict(), "model.pth")
    print("Best model saved as 'model.pth'")


No saved model found — starting training from scratch.

Epoch [1/15]
------------------------------


Train Loss: 0.0021 | Train Acc: 0.9993
Val   Loss: 2.3934 | Val   Acc: 0.7925
Validation accuracy improved — saving best model.

Epoch [2/15]
------------------------------


Train Loss: 0.0067 | Train Acc: 0.9980
Val   Loss: 2.3644 | Val   Acc: 0.8005
Validation accuracy improved — saving best model.

Epoch [3/15]
------------------------------


Train Loss: 0.0043 | Train Acc: 0.9985
Val   Loss: 1.9039 | Val   Acc: 0.8035
Validation accuracy improved — saving best model.

Epoch [4/15]
------------------------------


Train Loss: 0.0012 | Train Acc: 0.9996
Val   Loss: 2.0386 | Val   Acc: 0.8035
No improvement for 1 epoch(s).

Epoch [5/15]
------------------------------


Train Loss: 0.0035 | Train Acc: 0.9986
Val   Loss: 2.0858 | Val   Acc: 0.8045
Validation accuracy improved — saving best model.

Epoch [6/15]
------------------------------


... (6 outputs left out)

Train Loss: 0.0023 | Train Acc: 0.9991
Val   Loss: 1.7207 | Val   Acc: 0.8080
No improvement for 4 epoch(s).

Epoch [13/15]
------------------------------


Train Loss: 0.0018 | Train Acc: 0.9994
Val   Loss: 1.8956 | Val   Acc: 0.8015
No improvement for 5 epoch(s).

Epoch [14/15]
------------------------------


Train Loss: 0.0011 | Train Acc: 0.9995
Val   Loss: 1.9184 | Val   Acc: 0.8005
No improvement for 6 epoch(s).

Epoch [15/15]
------------------------------


Train Loss: 0.0097 | Train Acc: 0.9968
Val   Loss: 1.8633 | Val   Acc: 0.7995
No improvement for 7 epoch(s).

Training complete. Best Validation Accuracy: 0.8105
Best model saved as 'model.pth'


In [56]:
class FeatureExtractor(nn.Module):
    def __init__(self, trained_model):
        super(FeatureExtractor, self).__init__()
        # Copy layers up to layer3
        self.features = nn.Sequential(
            trained_model.conv1,
            trained_model.bn1,
            trained_model.relu,
            trained_model.maxpool,
            trained_model.layer1,
            trained_model.layer2,
            trained_model.layer3
        )

    def forward(self, x):
        with torch.no_grad():  # inference only
            x = self.features(x)
        return x

# Instantiate and show model in eval mode
feature_extractor = FeatureExtractor(model).to(device)
feature_extractor.eval()

print("\nFeature Extractor (up to layer3):\n")
print(feature_extractor)



Feature Extractor (up to layer3):

FeatureExtractor(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (4): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, moment